# Quantization - Production Implementation

**Complete guide with real HuggingFace libraries and production patterns.**

This notebook uses:
- Real models from HuggingFace Hub
- Production-grade patterns
- Error handling and optimization
- Real-world use cases

See also: `llm/concepts/quantization.md` for theory and interview Q&A

## Setup

In [ ]:
# Install required packages
# !pip install transformers torch sentence-transformers datasets peft bitsandbytes

import warnings
warnings.filterwarnings('ignore')

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

## Quick Start

In [ ]:
# Model Quantization with Transformers
from transformers import AutoModelForSequenceClassification, quantization_aware_training
import torch

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased")

# Convert to int8
model = model.to(torch.int8)

# Or use bitsandbytes for 8-bit inference
from bitsandbytes.nn import Linear8bitLt

# Quantized linear layer
# quantized_layer = Linear8bitLt(768, 3072)

## Production Implementation

In [ ]:
# Production Quantization Pipeline
from transformers import AutoModelForSequenceClassification
import torch
from torch.quantization import quantize_dynamic, QConfig, MinMaxObserverWithHistogram

class QuantizationService:
    """Production model quantization"""

    def __init__(self, model_name="bert-base-uncased"):
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)

    def quantize_dynamic(self):
        """Dynamic quantization (easiest)"""
        quantized = quantize_dynamic(
            self.model,
            {torch.nn.Linear},
            dtype=torch.qint8
        )
        return quantized

    def get_size_reduction(self, original_model, quantized_model):
        """Compare sizes"""
        original_size = sum(p.numel() * 4 for p in original_model.parameters()) / 1e6
        quantized_size = sum(p.numel() for p in quantized_model.parameters()) / 1e6

        return {
            "original_mb": original_size,
            "quantized_mb": quantized_size,
            "reduction": original_size / quantized_size
        }

# Usage
service = QuantizationService()
# quantized = service.quantize_dynamic()
# sizes = service.get_size_reduction(service.model, quantized)
# print(f"Reduction: {sizes['reduction']:.1f}x")

## Real-World: Onnx

In [ ]:
# Real-World: ONNX Quantization
from transformers import AutoModelForSequenceClassification, AutoTokenizer

class ONNXQuantizationService:
    """Export to ONNX and quantize"""

    def __init__(self, model_name="distilbert-base-uncased"):
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

    def export_to_onnx(self, output_path="model.onnx"):
        """Export model to ONNX format"""
        try:
            from torch.onnx import export
            dummy_input = self.tokenizer("sample", return_tensors="pt")

            # export(
            #     self.model,
            #     tuple(dummy_input.values()),
            #     output_path,
            #     input_names=['input_ids', 'attention_mask'],
            #     output_names=['logits']
            # )

            print(f"Exported to {output_path}")
        except Exception as e:
            print(f"ONNX export: {e}")

    def quantize_onnx(self, model_path, output_path):
        """Quantize ONNX model"""
        try:
            from onnxruntime.quantization import quantize_dynamic, QuantType

            # quantize_dynamic(model_path, output_path, weight_type=QuantType.QInt8)
            print(f"Quantized to {output_path}")
        except Exception as e:
            print(f"Quantization: {e}")

# Usage
service = ONNXQuantizationService()
# service.export_to_onnx("model.onnx")
# service.quantize_onnx("model.onnx", "model-quantized.onnx")

## Production Checklist

- [ ] Load models from HuggingFace Hub
- [ ] Set up GPU device handling
- [ ] Implement batch processing
- [ ] Add error handling
- [ ] Optimize for latency
- [ ] Add logging and monitoring
- [ ] Test with production data
- [ ] Create inference service

## Useful Links

- [HuggingFace Models](https://huggingface.co/models)
- [HuggingFace Documentation](https://huggingface.co/docs/transformers)
- [PEFT Library](https://github.com/huggingface/peft)
- [Sentence Transformers](https://www.sbert.net/)